# SIMT 同步机制

## 概述

前面几节我们已经能写出一个基础的 SIMT 算子。但当多个线程需要**协作**完成一件事——比如先一起把数据写进共享内存、再统一读取——线程之间的执行先后和内存可见性就成了正确性的关键。本节介绍 SIMT 编程中的同步机制，帮助你在这类场景下写出正确的并行代码。

### 前置要求

- 已理解 Thread Block 内多个线程可以访问共享内存，也可能同时访问 Global Memory。
- 了解数据竞争、执行顺序和内存可见性是并行程序正确性的关键问题。
- 本小节为理论讲解，不依赖在线硬件环境。

### 学习目标

学完本小节后，你应该能够：

- 解释为什么 SIMT 多线程程序需要同步机制。
- 区分线程同步和内存可见性顺序这两类问题。
- 说明 `asc_syncthreads`、`asc_threadfence`、`asc_threadfence_block` 的作用和适用范围。
- 判断共享内存协作、全局内存写入和线程块内通信分别应该关注哪类同步。
- 理解同步机制对正确性和性能的影响。

### 小节内容

- 为什么需要同步
- 先区分两类同步问题
- 同步接口总览
- `asc_syncthreads`：线程块内等待
- `asc_threadfence`：内存写入顺序保证
- `asc_threadfence_block`：线程块内内存可见性
- 怎么选择同步接口
- 常见误区

### 为什么需要同步

SIMT 是单指令多线程的编程模型。多个线程并发执行时，线程之间通常相互独立，但只要它们存在数据依赖或数据冲突，就可能出现顺序和可见性问题。

典型场景包括：

- 计算分两个阶段，先用多个线程把数据写入共享内存，然后再由其它线程读取，以提升热点数据的访存效率。
- 多个线程对同一片 Global Memory 或共享内存区域进行读写。
- 一个线程要保证写入数据的顺序，其它线程据此读取数据。

如果没有同步机制，线程可能在数据尚未准备好时就开始读取，或者看到的内存操作顺序和程序预期不一致，从而导致数据竞争或计算结果错误。因此，Ascend C 提供了一组同步接口，让开发者按需选择合适的机制来保证异步操作的正确性。

### 先区分两类同步问题

学习同步接口前，先区分两个容易混淆的问题：

| 问题类型 | 关心什么 | 典型提问 |
| --- | --- | --- |
| 线程执行位置同步 | 线程是否都执行到了某个位置 | 其它线程是否都写完共享内存了？ |
| 内存操作顺序/可见性 | 某个线程之前的读写是否对其它线程可见 | 我写入的数据，别的线程能按预期看到吗？ |

`asc_syncthreads` 主要解决第一类问题：让当前线程块内所有线程都等待到同一位置。

`asc_threadfence` 和 `asc_threadfence_block` 主要解决第二类问题：保证内存读写操作的顺序和可见性。

### 同步接口总览

Ascend C 为 SIMT 场景提供了以下同步接口：

| 接口名 | 功能说明 | 是否阻塞线程 | 作用范围 |
| --- | --- | --- | --- |
| `asc_syncthreads` | 等待当前线程块内所有线程代码都执行到该函数位置 | 是 | 当前 Thread Block 内的线程 |
| `asc_threadfence` | 保证不同线程对同一份全局、共享内存访问过程中，写入操作的时序性；不阻塞线程，仅保证内存操作可见性顺序 | 否 | 所有线程 |
| `asc_threadfence_block` | 协调同一 Thread Block 内线程之间的内存操作顺序，确保调用前的读写对同一线程块内其它线程可见 | 否 | 当前 Thread Block 内的线程 |

### `asc_syncthreads`：线程块内等待

`asc_syncthreads` 的作用是等待当前 Thread Block 内所有 thread 都执行到该函数位置。它是一个阻塞式同步点。

最典型的使用场景是线程块内共享内存协作：

```cpp
// 阶段 1：线程块内多个线程写入共享内存
shared_buf[threadIdx.x] = input[idx];

// 等待同一线程块内所有线程都完成写入
asc_syncthreads();

// 阶段 2：线程块内线程读取共享内存进行后续计算
float v = shared_buf[some_index];
```

教学上可以把它理解成一个集合点：线程块里的线程必须都走到这里，才能继续向后执行。

### `asc_threadfence`：内存写入顺序保证

**先看一个线程间通信的场景**。当一个线程想与另一个线程通信时，通常可以借助一个内存变量（标记位）来实现：一个线程写，另一个线程读。同一个线程块内的线程可以通过共享内存通信，运行在不同线程块内的线程则可以通过 Global Memory 通信。

但这里有一个隐患：**线程的执行顺序是乱序的，此外编译器对代码的优化也可能导致指令乱序执行**。设想这样一对线程：

- 线程 1：先写入数据，再修改内存标记位，表示「数据已就绪」。
- 线程 2：轮询标记位，一旦发现被修改，就去读取线程 1 写入的数据。

如果不加约束，线程 1 的「写数据」和「改标记位」两步可能被乱序——标记位先被改了，数据却还没写完。线程 2 看到标记位后读到的就是未就绪的脏数据。

解决办法是：**让线程 1 在「写数据」和「改标记位」之间插入一道内存栅栏**，强制保证写数据一定先于改标记位对其它线程可见。这正是 `asc_threadfence` 的用途。用伪代码表示如下：

```cpp
// 线程 1：生产数据
data = produce();          // ① 写入数据
asc_threadfence();         // ② 内存栅栏：保证 ① 先于 ③ 对其它线程可见
flag = 1;                  // ③ 修改标记位，宣告“数据已就绪”

// 线程 2：消费数据
while (flag != 1) { }      // ④ 轮询，直到标记位被置 1
result = consume(data);    // ⑤ 此时再读 data，一定是 ① 写入的有效数据
```

如果去掉第 ② 行的栅栏，③ 可能先于 ① 生效，线程 2 在第 ④ 行看到 `flag == 1` 后，第 ⑤ 行读到的 `data` 却还是旧值。栅栏的作用就是堵住这个乱序的缺口。

`asc_threadfence` 用于保证不同线程对同一份全局、共享内存访问过程中，写入操作的时序性。它不会阻塞线程，仅保证内存操作的可见性顺序。

需要特别注意：

- 它不是“等所有线程都到这里”。
- 它关注的是调用前后的内存操作顺序。
- 它适合用于需要保证写入结果按预期被其它线程观察到的场景。

可以把 `asc_threadfence` 理解为内存层面的栅栏：它不负责让线程排队，但负责约束内存操作的先后可见关系。

### `asc_threadfence_block`：线程块内内存可见性

`asc_threadfence_block` 用于协调同一 Thread Block 内线程之间的内存操作顺序。它确保某一线程在调用 `asc_threadfence_block()` 之前的所有内存操作，对同一线程块内的其它线程是可见的。

它和 `asc_threadfence` 的区别可以这样理解：

| 接口 | 重点 | 范围理解 |
| --- | --- | --- |
| `asc_threadfence` | 更一般的内存访问时序性 | 面向全局、共享内存访问过程 |
| `asc_threadfence_block` | 线程块内线程之间的内存操作顺序 | 当前 Thread Block 内 |

如果问题明确发生在线程块内部，优先关注 `asc_threadfence_block` 和 `asc_syncthreads` 的配合。

### 怎么选择同步接口

可以按下面的问题来选择：

| 你想解决的问题 | 优先考虑 | 原因 |
| --- | --- | --- |
| 线程块内所有线程必须都写完共享内存，才能继续读 | `asc_syncthreads` | 需要阻塞等待所有线程到达同步点 |
| 需要保证写入操作的可见顺序，但不需要等待所有线程 | `asc_threadfence` | 关注内存操作顺序，不阻塞线程 |
| 只需要保证同一线程块内调用前读写对其它线程可见 | `asc_threadfence_block` | 作用范围是当前 Thread Block |

一个实用判断：如果你担心“其它线程还没执行到这里”，看 `asc_syncthreads`；如果你担心“其它线程看不到我之前的写入”，看 `threadfence` 系列。

### 常见误区

| 误区 | 正确认知 |
| --- | --- |
| `asc_threadfence` 会等待其它线程 | 不会。它不阻塞线程，只保证内存操作可见性顺序。 |
| 使用共享内存就一定安全 | 不一定。共享内存被线程块内多个线程访问，仍然可能出现数据竞争。 |
| 只要用了同步接口，性能一定更好 | 不一定。同步用于保证正确性，过度同步可能降低并行效率。 |
| `asc_syncthreads` 可以跨线程块同步 | 不可以。它等待当前线程块内所有线程。 |

同步接口的核心价值是让并发程序的执行顺序符合你的数据依赖，而不是给所有代码都加保险。

### 术语速查

| 术语 | 说明 |
| --- | --- |
| 数据竞争 | 多个线程访问同一内存区域、且至少一个线程写入，而访问顺序未被正确约束的情形。 |
| 线程同步 | 让多个线程在执行位置上达成某种等待关系。 |
| 内存可见性 | 一个线程的内存写入能否被其它线程按预期观察到。 |
| `asc_syncthreads` | 等待当前线程块内所有线程到达调用位置，属阻塞式同步。 |
| `asc_threadfence` | 保证全局/共享内存访问过程中写入操作的时序性，不阻塞线程。 |
| `asc_threadfence_block` | 保证调用前的内存操作对同一线程块内其它线程可见。 |

## 小节小结

本小节介绍了 SIMT 编程中的同步机制。`asc_syncthreads` 用于等待当前线程块内所有线程到达同一位置，适合共享内存协作场景；`asc_threadfence` 用于保证全局、共享内存访问过程中写入操作的时序性，不阻塞线程；`asc_threadfence_block` 用于保证调用前的内存读写对同一线程块内其它线程可见。

写 SIMT 程序时，同步机制首先服务于正确性：当线程之间存在数据依赖时，需要选择合适的同步方式；当线程之间没有依赖时，过度同步反而可能削弱并行效率。

到这里，SIMT 编程模型的核心概念已经讲完。这些同步接口，连同前面的核函数、内置变量、内存操作等，都属于 Ascend C 为 SIMT 编程提供的 API。下一章节《编程 API》会把这些 API 按用途整理成一张完整的学习地图，方便你在真实开发中快速定位。

## 课后练习

本节介绍了 SIMT 同步机制中线程同步与内存可见性两类问题，以及 `asc_syncthreads`、`asc_threadfence`、`asc_threadfence_block` 三个接口，请根据学习内容完成以下题目进行自测。

1. （判断题）`asc_threadfence` 会阻塞线程，直到当前线程块内所有线程都执行到该调用位置。

2. （单选题）`asc_syncthreads` 的作用是什么？  
    A. 保证内存写入的可见性顺序，但不阻塞线程  
    B. 等待当前线程块内所有线程都执行到该函数位置  
    C. 等待整个 Grid 内所有线程到达同步点  
    D. 为线程块分配共享内存  

3. （单选题）多个线程先把数据写入共享内存，再统一读取共享内存进行后续计算，应优先使用哪个接口来保证「所有线程都写完才开始读」？  
    A. `asc_threadfence`  
    B. `asc_threadfence_block`  
    C. `asc_syncthreads`  
    D. 无需任何同步  

4. （多选题）以下关于 SIMT 同步机制的说法，哪些是正确的？  
    A. `asc_syncthreads` 不阻塞线程，调用后线程立即继续向后执行  
    B. `asc_threadfence` 不阻塞线程，只保证内存写入操作的可见性顺序  
    C. `asc_threadfence_block` 确保调用前的内存操作对同一线程块内其它线程可见  
    D. 同步用于保证正确性，过度同步反而可能降低并行效率  

**执行以下代码获取答案。**

## 参考答案

运行下面的单元格查看课后练习参考答案。


In [ ]:
!cat answer/03.04.05_answer.txt
